# Variable labels

Collects the variable labels from every XPT header in `$NHANES_DATA` → `$NHANES_WORK/variable_labels.csv` (module, cycle, variable, label). First notebook of the pipeline; `run_pipeline.sh` runs them in order.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "nhanes" / "paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("run this notebook from inside the repository")
sys.path.insert(0, str(ROOT))
from nhanes.paths import DATA, WORK  # noqa: E402

In [2]:
SUF = {"_D": "2005-06", "_E": "2007-08", "_F": "2009-10", "_G": "2011-12", "_H": "2013-14",
       "_I": "2015-16", "_J": "2017-18"}
rows = []
for p in sorted(p for p in DATA.glob("*/*") if p.suffix.lower() == ".xpt"):
    stem = p.stem.upper()
    comp, cyc = (stem[2:], "2017-20P") if stem.startswith("P_") else (stem[:-2], SUF.get(stem[-2:]))
    if cyc is None:
        continue
    for f in pd.read_sas(p, format="xport", iterator=True).fields:
        rows.append((comp, cyc, f["name"].decode().strip(), f["label"].decode("latin1").strip()))
labels = pd.DataFrame(rows, columns=["module", "cycle", "variable", "label"])
labels.to_csv(WORK / "variable_labels.csv", index=False)

In [3]:
print(f"{len(labels):,} (module, cycle, variable) labels from {labels.groupby(['module', 'cycle']).ngroups} files")
display(labels.groupby("cycle").agg(files=("module", "nunique"), variables=("variable", "size")))

8,717 (module, cycle, variable) labels from 322 files


,files,variables
cycle,,
2005-06,40,1163
2007-08,40,1140
2009-10,40,1089
2011-12,43,1140
2013-14,44,1182
2015-16,40,1149
2017-18,39,1034
2017-20P,36,820
